# PDCP · N13 v2 — Larger models: does scale fix stress perception?
**Kaggle:** **GPU T4 x2** · Internet **On** · **Save & Run All (Commit)**. v2: Pixtral-12B is split across both GPUs (the first run ran out of memory on one GPU); set `RESUME_ROOT` to the first run's output so InternVL3.5-14B is not repeated. Expected ≈ 2–3 h.

**Inputs:** `kbs-n3-output` (pairs, drawings, images) · `kbs-n5-output` (E1 answers, for comparison) · `kbs-n6-part2` (MEL counterfactual images + E2 answers).

**Same protocol as E4b (N10):** the identical 200 base pairs (both orders) + their mean-edge-length counterfactuals (δ −0.05 / +0.10), prompt P-Def.
- **Part A — larger open-weight models (GPU):** InternVL3.5-14B and Pixtral-12B (a different model family), 4-bit NF4 weights, one model per GPU in parallel.
- **Part B — proprietary model (optional):** a current Gemini Flash model through the Gemini API free tier. Runs only if a Kaggle secret `GEMINI_API_KEY` exists (Add-ons → Secrets). Free-tier daily limits apply, so this part stops at `API_MAX_PER_RUN` and continues where it stopped on the next run (set `RESUME_ROOT`).

**Outputs** (`/kaggle/working/n13_output/`): `results/e5/<model>.jsonl`, `e5_summary.csv` (same columns as `e4_size_summary.csv`), `e5_status.json`, `fig_e5_scale.png`, `pdcp_infer.py` v0.6, logs, `MANIFEST.txt`.

In [1]:
# ==== SETTINGS ====
N03_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n3-output"
N05_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n5-output"
N06_ROOT = "/kaggle/input/datasets/resadussalehinrafsan/kbs-n6-part2"
RESUME_ROOT = "/kaggle/input/datasets/rsrafsan/kbs-n13-v1-output"   # first run (InternVL-14B done); set None for a fresh start
OUTPUT_DIR = "/kaggle/working/n13_output"
N_BASE = 200
SEED = 2026
MEL_DELTAS = [-0.05, 0.10]
GPU_MODELS = ["InternVL-14B", "Pixtral-12B"]
RUN_API = True                     # Part B; skipped automatically if the secret GEMINI_API_KEY is missing
GEMINI_MODEL = "auto"              # "auto" = newest available Flash model with image input; or give an exact model id
API_RPM = 8                        # requests per minute (stay below the free-tier limit)
API_MAX_PER_RUN = 500              # stop after this many requests in one run (daily limit); rerun with RESUME_ROOT to continue
SESSION_HOURS = 11.3
DRY_RUN = False

In [2]:
import os, sys, json, time, glob, shutil, subprocess, importlib, re
T0 = time.time()
for d in ["results/e5", "logs"]: os.makedirs(f"{OUTPUT_DIR}/{d}", exist_ok=True)
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
PARAMS_B = {"Q3VL-2B": 2, "Q3VL-4B": 4, "Q25VL-7B": 7, "InternVL-4B": 4, "InternVL-14B": 14, "Pixtral-12B": 12, "Gemini-Flash": np.nan}
FAMILY = {"Q3VL-2B": "Qwen3-VL", "Q3VL-4B": "Qwen3-VL", "Q25VL-7B": "Qwen2.5-VL", "InternVL-4B": "InternVL3.5", "InternVL-14B": "InternVL3.5",
          "Pixtral-12B": "Pixtral", "Gemini-Flash": "Gemini (proprietary)"}
if not DRY_RUN:
    import torch
    N_GPU = torch.cuda.device_count(); assert N_GPU >= 1, "Set Accelerator = GPU T4 x2"
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.57", "accelerate", "bitsandbytes", "qwen-vl-utils", "num2words", "google-genai"], capture_output=True, text=True)
else:
    N_GPU = 2
def find(root, name):
    f = sorted(glob.glob(f"{root}/**/{name}", recursive=True)); assert f, f"{name} not found under {root}"; return os.path.dirname(f[0])
N03 = find(N03_ROOT, "pairs.parquet"); N06 = find(N06_ROOT, "mel_cf_pairs.parquet")
shutil.copy(f"{N06}/pdcp_utils.py", f"{OUTPUT_DIR}/pdcp_utils.py"); sys.path.insert(0, OUTPUT_DIR); import pdcp_utils as U
PR = pd.read_parquet(f"{N03}/pairs.parquet"); DR = pd.read_parquet(f"{N03}/drawings.parquet"); MCP = pd.read_parquet(f"{N06}/mel_cf_pairs.parquet")
png = dict(zip(DR.drawing_id, DR.png))
def path(did): return f"{N03}/{png[did]}" if did in png else f"{N06}/images/768/{did}.png"
# identical base selection to N08 / N10
SUB = PR[(PR.set == "main") & PR.subset400].copy()
BASE = SUB.groupby(["dstress_design", "cond"], group_keys=False).sample(frac=min(1.0, N_BASE / len(SUB)), random_state=SEED)
if len(BASE) < N_BASE: BASE = pd.concat([BASE, SUB.drop(BASE.index).sample(min(N_BASE - len(BASE), len(SUB) - len(BASE)), random_state=SEED)])
BASE = BASE.head(N_BASE)
MEL = MCP[MCP.base_pair.isin(BASE.pair_id) & MCP.cf_delta.isin(MEL_DELTAS) & (MCP.cf_proxy == "mel")]
rows = []
for df, kind in [(BASE, "base"), (MEL, "mel")]:
    for r in df.itertuples():
        a, b = path(r.drawing_A), path(r.drawing_B)
        for order in ("AB", "BA"):
            img1, img2 = (a, b) if order == "AB" else (b, a)
            corr = "same" if r.correct == "same" else ("1" if (r.correct == "A") == (order == "AB") else "2")
            rows.append({"pair_id": r.pair_id, "order": order, "prompt_id": "P-Def", "img1": img1, "img2": img2, "correct_img": corr, "kind": kind})
Q = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
Q.to_csv(f"{OUTPUT_DIR}/e5_queries.csv", index=False)
if not DRY_RUN: assert all(os.path.exists(p) for p in set(Q.img1) | set(Q.img2)), "missing images"
print(len(BASE), "base pairs |", len(MEL), "MEL pairs |", len(Q), "queries per model")
if RESUME_ROOT and os.path.exists(RESUME_ROOT):
    for f in glob.glob(f"{RESUME_ROOT}/**/results/e5/*.jsonl", recursive=True):
        shutil.copy(f, f"{OUTPUT_DIR}/results/e5/{os.path.basename(f)}"); print("resumed", os.path.basename(f))

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

200 base pairs | 384 MEL pairs | 1168 queries per model
resumed InternVL-14B.jsonl


## Inference worker v0.6

In [3]:
%%writefile /kaggle/working/n13_output/pdcp_infer.py
"""
pdcp_infer.py — PDCP VLM inference worker (v0.6.0). One model per process.
v0.6.1: models can span both GPUs ("multi"), SDPA attention; a query that runs out of memory is retried once with images downscaled to 512 px (flagged "downscaled").
v0.6: larger models (InternVL3.5-14B, Pixtral-12B), 4-bit.
v0.5: more model sizes (Q25VL-3B, InternVL-2B/8B), 8-bit loading, batch cap per prompt (few-shot = 1), longer outputs for P-NoReason/P-ReasonFirst.
v0.4: per-row prompt_id (many prompt variants per process), per-prompt max_new_tokens, few-shot examples (--fewshot).
v0.3: processor_kwargs API, per-model processor options + default batch, 25-word reasons, early abort on unparseable output.

Usage:
  python pdcp_infer.py --model Q3VL-4B --pairs pairs.csv --out results.jsonl --summary sum.json
                       [--prompt P-Def] [--batch 1] [--batch_test 4] [--max_new 160] [--hf_token TOKEN]

pairs.csv columns: pair_id, order (AB|BA), img1, img2 (absolute PNG paths), correct_img ("1"|"2"|"same")
Resumable: rows already in --out (same pair_id+order+prompt) are skipped.
"""
import argparse, gc, json, os, re, sys, time, traceback

MODELS = {   # batch = default batch size on one T4 (from the N04 pilot); proc = extra processor options
    "Q3VL-2B":     {"ids": ["Qwen/Qwen3-VL-2B-Instruct"], "quant": None, "batch": 8, "proc": {}},
    "Q3VL-4B":     {"ids": ["Qwen/Qwen3-VL-4B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q3VL-8B":     {"ids": ["Qwen/Qwen3-VL-8B-Instruct"], "quant": "nf4", "batch": 2, "proc": {}},
    "Q25VL-7B":    {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "nf4", "batch": 4, "proc": {}},
    "Gemma3-4B":   {"ids": ["google/gemma-3-4b-it"], "quant": None, "batch": 4, "proc": {}},
    "InternVL-4B": {"ids": ["OpenGVLab/InternVL3_5-4B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4,
                    "proc": {"crop_to_patches": False}},   # one 448-px tile per image (pilot: tiling -> 11 s/query, OOM at batch 4)
    "SmolVLM-2B":  {"ids": ["HuggingFaceTB/SmolVLM2-2.2B-Instruct"], "quant": None, "batch": 1, "proc": {}},
    "InternVL-14B": {"ids": ["OpenGVLab/InternVL3_5-14B-HF", "OpenGVLab/InternVL3-14B-hf"], "quant": "nf4", "batch": 2, "proc": {"crop_to_patches": False}},
    "Pixtral-12B": {"ids": ["mistral-community/pixtral-12b"], "quant": "nf4", "batch": 1, "proc": {}, "multi": True},
    "Q25VL-3B":    {"ids": ["Qwen/Qwen2.5-VL-3B-Instruct"], "quant": None, "batch": 4, "proc": {}},
    "Q25VL-7B-int8": {"ids": ["Qwen/Qwen2.5-VL-7B-Instruct"], "quant": "int8", "batch": 2, "proc": {}},
    "InternVL-2B": {"ids": ["OpenGVLab/InternVL3_5-2B-HF", "OpenGVLab/InternVL3-2B-hf"], "quant": None, "batch": 4, "proc": {"crop_to_patches": False}},
    "InternVL-8B": {"ids": ["OpenGVLab/InternVL3_5-8B-HF", "OpenGVLab/InternVL3-8B-hf"], "quant": "nf4", "batch": 2, "proc": {"crop_to_patches": False}},
}

SYSTEM = ("You are a careful visual analyst. You will be shown two images of node-link diagrams that depict the SAME "
          "network (same nodes and same edges), drawn with different node positions. Answer the question strictly in "
          "the JSON format requested. Do not add any text outside the JSON.")
DEFINITION = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures "
              "how well the straight-line distances between pairs of nodes match the graph-theoretic distances between them "
              "(the number of edges on the shortest path connecting them). A drawing has LOW stress when pairs of nodes that "
              "are few edges apart are drawn close together and pairs that are many edges apart are drawn far apart, "
              "consistently across the whole drawing. A drawing has HIGH stress when these distances disagree.")
JSON_SPEC = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>, '
             '"reason": "<at most 25 words>"}')
QUESTION = "\n\nImage 1 and Image 2 show the same network. Which image has LOWER stress?\n\n"
DEF_NEUTRAL = ("In a node-link diagram, each circle is a node and each line is an edge. The \"stress\" of a drawing measures how well "
               "the straight-line distances between ALL pairs of nodes are proportional to their graph-theoretic distances (the number of "
               "edges on the shortest path between them). Only this proportionality matters: the overall size of the drawing, how compact "
               "or spread out it looks, and the absolute length of the edges do not affect stress. A drawing has low stress when the ratio "
               "between drawn distance and path distance is the same for all pairs of nodes.")
STRUCT = ("\n\nEvaluate each image on three criteria, then decide:\n1. Node distribution: are nodes spread evenly, or clumped?\n"
          "2. Edge lengths: are edges of similar length, or do lengths vary a lot?\n3. Edge crossings: how many edges cross each other?\n"
          "Drawings with evenly spread nodes, uniform edge lengths and few crossings usually have lower stress, but judge the overall "
          "correspondence between distances and path lengths.")
J_STRUCT = ('Respond with JSON only:\n{"criteria": {"image1": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}, '
            '"image2": {"node_distribution": "<short>", "edge_lengths": "<short>", "crossings": "<short>"}}, '
            '"answer": "1" | "2" | "same", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_NODIFF = ('Respond with JSON only:\n{"answer": "1" | "2" | "no clear difference", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_2AFC = ('You must choose one image. Respond with JSON only:\n{"answer": "1" | "2", "confidence": <integer 1-5>, "reason": "<at most 25 words>"}')
J_RFIRST = ('First think step by step in the "reason" field, then give the answer. Respond with JSON only:\n'
            '{"reason": "<at most 60 words>", "answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
J_NOREASON = ('Respond with JSON only:\n{"answer": "1" | "2" | "same", "confidence": <integer 1-5>}')
PROMPTS = {
    "P-Def": DEFINITION + QUESTION + JSON_SPEC,
    "P-Expert": ("You are an expert in graph drawing and network visualization. Image 1 and Image 2 show the same network. "
                 "Which image has lower stress?\n\n" + JSON_SPEC),
    "P-NoDef": QUESTION.strip() + "\n\n" + JSON_SPEC,
    "P-DefNeutral": DEF_NEUTRAL + QUESTION + JSON_SPEC,
    "P-Struct": DEFINITION + STRUCT + QUESTION + J_STRUCT,
    "P-NoDiff": DEFINITION + QUESTION + J_NODIFF,
    "P-2AFC": DEFINITION + QUESTION + J_2AFC,
    "P-ReasonFirst": DEFINITION + QUESTION + J_RFIRST,
    "P-NoReason": DEFINITION + QUESTION + J_NOREASON,
    "P-FewShot3": DEFINITION + QUESTION + JSON_SPEC,     # + 3 worked examples inserted before the question
}
MAX_NEW = {"P-Struct": 230, "P-ReasonFirst": 300, "P-NoReason": 96}
BATCH_CAP = {"P-FewShot3": 1}   # 8 images per query: one query at a time (N08 ran out of memory)
FEWSHOT = []   # filled from --fewshot: list of {"img1","img2","answer_json"}


def parse_answer(text):
    out = {"answer": None, "confidence": None, "reason": None, "parse_ok": False}
    if not text:
        return out
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            a = str(obj.get("answer", "")).strip().lower().replace("image", "").strip()
            if a in ("1", "2"):
                out["answer"] = a
            elif a in ("same", "no clear difference"):
                out["answer"] = "same"
            c = obj.get("confidence")
            out["confidence"] = int(c) if c is not None and str(c).strip().isdigit() else None
            out["reason"] = obj.get("reason")
            out["parse_ok"] = out["answer"] is not None
            if out["parse_ok"]:
                return out
        except Exception:
            pass
    m = re.search(r'answer"?\s*(?:is)?\s*[:=]?\s*"?\s*(image\s*)?(1|2|same|no clear difference)\b', text, re.I)
    if m:
        a = m.group(2).lower()
        out["answer"] = "same" if a.startswith(("same", "no")) else a
        out["parse_ok"] = True
        r = re.search(r'"reason"\s*:\s*"([^"]*)', text)
        out["reason"] = r.group(1) if r else None
    return out


def build_messages(img1, img2, prompt_id, with_images=True):
    def im(x):
        return {"type": "image", "image": x} if with_images else {"type": "image"}
    content = [{"type": "text", "text": SYSTEM}]
    if prompt_id == "P-FewShot3" and FEWSHOT:
        content[0]["text"] += "\n\nFirst, three worked examples with the correct answers."
        for k, ex in enumerate(FEWSHOT, 1):
            content += [{"type": "text", "text": f"Example {k}. Image 1:"}, im(ex["img1_pil"]), {"type": "text", "text": "Image 2:"},
                        im(ex["img2_pil"]), {"type": "text", "text": "Correct answer: " + ex["answer_json"]}]
        content.append({"type": "text", "text": "Now the actual question."})
    content += [{"type": "text", "text": "Image 1:"}, im(img1), {"type": "text", "text": "Image 2:"}, im(img2),
                {"type": "text", "text": PROMPTS[prompt_id]}]
    return [{"role": "user", "content": content}]


class Runner:
    def __init__(self, key, hf_token=None, log=print):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        self.torch, self.key, self.log = torch, key, log
        spec = MODELS[key]
        self.route, self.model, self.proc, self.model_id, self.dtype_name = None, None, None, None, None
        errors = []
        for mid in spec["ids"]:
            for dt in (["float16", "bfloat16"] if spec["quant"] is None else ["float16"]):
                try:
                    t0 = time.time()
                    kw = {"device_map": "auto" if spec.get("multi") else {"": 0}, "token": hf_token, "low_cpu_mem_usage": True}
                    if spec.get("multi"): kw["attn_implementation"] = "sdpa"
                    if spec["quant"] == "int8":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
                    elif spec["quant"] == "nf4":
                        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16,
                                                                        bnb_4bit_use_double_quant=True)
                    else:
                        kw["dtype"] = getattr(torch, dt)
                    if self.model is None or self.model_id != mid:
                        self._free()
                        self.model = AutoModelForImageTextToText.from_pretrained(mid, **kw).eval()
                        self.proc = AutoProcessor.from_pretrained(mid, token=hf_token)
                    self.model_id, self.dtype_name, self.load_s = mid, (spec["quant"] or dt), time.time() - t0
                    return
                except Exception as e:
                    errors.append(f"{mid} {dt}: {type(e).__name__}: {str(e)[:300]}")
                    self.log("load failed: " + errors[-1])
                    self._free()
        raise RuntimeError("all loads failed | " + " || ".join(errors))

    def reload_dtype(self, dt):
        """Reload the same model in another dtype (used when fp16 produces garbage)."""
        from transformers import AutoModelForImageTextToText
        self._free(keep_proc=True)
        self.model = AutoModelForImageTextToText.from_pretrained(self.model_id, device_map={"": 0},
                                                                 dtype=getattr(self.torch, dt), low_cpu_mem_usage=True).eval()
        self.dtype_name = dt

    def _free(self, keep_proc=False):
        if self.model is not None:
            del self.model
            self.model = None
        if not keep_proc:
            self.proc = None
        gc.collect()
        try:
            self.torch.cuda.empty_cache()
        except Exception:
            pass

    def _inputs(self, batch, prompt_id):
        """batch = list of (img1, img2). Tries route A (chat template loads images) then B (processor call)."""
        torch = self.torch
        routes = [self.route] if self.route else ["A", "B_nested", "B_flat"]
        last = None
        for route in routes:
            try:
                if route == "A":
                    msgs = [build_messages(a, b, prompt_id) for a, b in batch]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    pk = dict(MODELS[self.key].get("proc", {}))
                    if len(batch) > 1:
                        pk["padding"] = True
                    inp = self.proc.apply_chat_template(msgs if len(batch) > 1 else msgs[0], add_generation_prompt=True,
                                                        tokenize=True, return_dict=True, return_tensors="pt",
                                                        processor_kwargs=pk)
                else:
                    texts = [self.proc.apply_chat_template(build_messages(a, b, prompt_id, with_images=False),
                                                           add_generation_prompt=True, tokenize=False) for a, b in batch]
                    ex = [x for e in FEWSHOT for x in (e["img1_pil"], e["img2_pil"])] if prompt_id == "P-FewShot3" else []
                    imgs = [ex + [a, b] for a, b in batch] if route == "B_nested" else [x for ab in batch for x in ex + list(ab)]
                    if len(batch) > 1:
                        self.proc.tokenizer.padding_side = "left"
                    inp = self.proc(text=texts, images=imgs, return_tensors="pt", padding=True, **MODELS[self.key].get("proc", {}))
                self.route = route
                dev = self.model.device
                mdt = torch.float16 if self.dtype_name in ("nf4", "int8", "float16") else getattr(torch, self.dtype_name)
                return {k: (v.to(dev, dtype=mdt) if torch.is_floating_point(v) else v.to(dev)) if hasattr(v, "to") else v
                        for k, v in inp.items()}
            except Exception as e:
                last = e
                if self.route:
                    raise
        raise RuntimeError(f"no input route worked: {type(last).__name__}: {last}")

    def generate(self, batch, prompt_id, max_new=160):
        torch = self.torch
        inp = self._inputs(batch, prompt_id)
        n_in = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = self.model.generate(**inp, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None)
        texts = self.proc.batch_decode(out[:, n_in:], skip_special_tokens=True)
        return texts, int(n_in)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True); ap.add_argument("--pairs", required=True)
    ap.add_argument("--out", required=True); ap.add_argument("--summary", required=True)
    ap.add_argument("--prompt", default="P-Def"); ap.add_argument("--batch", type=int, default=0, help="0 = model default")
    ap.add_argument("--min_parse", type=float, default=0.5, help="abort if parse rate after 40 queries is below this")
    ap.add_argument("--deadline", type=float, default=0, help="unix time after which the worker stops cleanly")
    ap.add_argument("--batch_test", type=int, default=0); ap.add_argument("--max_new", type=int, default=160)
    ap.add_argument("--hf_token", default=None); ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--fewshot", default=None, help="json list of {img1,img2,answer_json}")
    a = ap.parse_args()
    import pandas as pd, torch
    from PIL import Image
    global FEWSHOT
    def log(s): print(f"[{a.model} {time.strftime('%H:%M:%S')}] {s}", flush=True)
    summ = {"model": a.model, "prompt": a.prompt, "ok": False, "error": None}
    t_start = time.time()
    try:
        P = pd.read_csv(a.pairs)
        if a.limit: P = P.head(a.limit)
        done = set()
        if os.path.exists(a.out):
            for line in open(a.out):
                try:
                    r = json.loads(line); done.add((r["pair_id"], r["order"], r["prompt_id"]))
                except Exception:
                    pass
        if "prompt_id" not in P.columns: P["prompt_id"] = a.prompt
        todo = [r for r in P.to_dict("records") if (r["pair_id"], r["order"], r["prompt_id"]) not in done]
        order_p = {p: i for i, p in enumerate(dict.fromkeys(P.prompt_id))}
        todo.sort(key=lambda r: order_p[r["prompt_id"]])            # stable: keeps the shuffle inside each prompt
        if a.fewshot:
            for e in json.load(open(a.fewshot)):
                e["img1_pil"] = Image.open(e["img1"]).convert("RGB"); e["img2_pil"] = Image.open(e["img2"]).convert("RGB"); FEWSHOT.append(e)
            log(f"loaded {len(FEWSHOT)} few-shot examples")
        log(f"{len(P)} pairs, {len(todo)} to do")
        torch.cuda.reset_peak_memory_stats()
        R = Runner(a.model, a.hf_token, log)
        log(f"loaded {R.model_id} as {R.dtype_name} in {R.load_s:.0f}s")
        summ.update({"model_id": R.model_id, "dtype": R.dtype_name, "load_s": round(R.load_s, 1)})
        load = lambda p: Image.open(p).convert("RGB")

        DOWN = set()
        # warm-up + dtype sanity: if fp16 gives zero parseable answers on 3 items, retry in bf16
        if todo and MODELS[a.model].get("multi"):
            log("multi-GPU model: warm-up check skipped")
        elif todo:
            w = todo[:3]
            texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in w[:1]], w[0]["prompt_id"], MAX_NEW.get(w[0]["prompt_id"], a.max_new))
            ok = parse_answer(texts[0])["parse_ok"]
            if not ok:
                t2 = [R.generate([(load(r["img1"]), load(r["img2"]))], r["prompt_id"], MAX_NEW.get(r["prompt_id"], a.max_new))[0][0] for r in w[1:]]
                if not any(parse_answer(t)["parse_ok"] for t in t2) and R.dtype_name == "float16":
                    log("fp16 output unparseable on warm-up -> retrying in bfloat16. sample: " + repr(texts[0][:200]))
                    try:
                        R.reload_dtype("bfloat16"); summ["dtype"] = "bfloat16"
                    except Exception as e:
                        log(f"bf16 reload failed ({type(e).__name__}); reloading fp16 and continuing")
                        R.reload_dtype("float16"); summ["dtype_note"] = "fp16 unparseable, bf16 unavailable"
        summ["route"] = R.route

        f = open(a.out, "a")
        times, n_ok, n_corr, n = [], 0, 0, 0
        bs = a.batch if a.batch > 0 else MODELS[a.model].get("batch", 1)
        summ["batch"] = bs
        groups, cur = [], []
        for r in todo:                       # chunks never mix prompt variants
            if cur and (r["prompt_id"] != cur[0]["prompt_id"] or len(cur) == min(bs, BATCH_CAP.get(cur[0]["prompt_id"], bs))):
                groups.append(cur); cur = []
            cur.append(r)
        if cur: groups.append(cur)
        for chunk in groups:
            pid = chunk[0]["prompt_id"]; mn = MAX_NEW.get(pid, a.max_new)
            if a.deadline and time.time() > a.deadline:
                log("deadline reached - stopping cleanly (resumable)"); summ["stopped_at_deadline"] = True; break
            t0 = time.time()
            def one_by_one(rows):
                out, nin = [], 0
                for r in rows:
                    try:
                        t, nin = R.generate([(load(r["img1"]), load(r["img2"]))], pid, mn); out += t
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        try:
                            small = lambda p: load(p).resize((512, 512))
                            t, nin = R.generate([(small(r["img1"]), small(r["img2"]))], pid, mn); out += t; DOWN.add((r["pair_id"], r["order"]))
                            log(f"OOM on {r['pair_id']} {r['order']} - answered with 512-px images")
                        except torch.cuda.OutOfMemoryError:
                            torch.cuda.empty_cache(); out.append(""); log(f"OOM on single query {r['pair_id']} {r['order']} - skipped (recorded as unparsed)")
                return out, nin
            if bs == 1 or len(chunk) == 1:
                texts, n_in = one_by_one(chunk)
            else:
                try:
                    texts, n_in = R.generate([(load(r["img1"]), load(r["img2"])) for r in chunk], pid, mn)
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); log("OOM at batch %d -> batch 1 from now on" % bs); bs = 1
                    texts, n_in = one_by_one(chunk)
            dt = (time.time() - t0) / len(chunk)
            for r, t in zip(chunk, texts):
                p = parse_answer(t)
                corr = p["answer"] == str(r["correct_img"]) if p["parse_ok"] else None
                rec = {"pair_id": r["pair_id"], "order": r["order"], "model": a.model, "model_id": R.model_id,
                       "dtype": R.dtype_name, "prompt_id": pid, "raw_text": t, "parsed": p, "parse_ok": p["parse_ok"],
                       "correct_img": str(r["correct_img"]), "correct": corr, "latency_s": round(dt, 3), "input_tokens": n_in,
                       "batch": len(chunk), "downscaled": (r["pair_id"], r["order"]) in DOWN, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
                f.write(json.dumps(rec) + "\n")
                n += 1; n_ok += p["parse_ok"]; n_corr += bool(corr)
            f.flush(); times.append(dt)
            if n >= 40 and n - len(chunk) < 40 and n_ok / n < a.min_parse:
                log(f"parse rate {n_ok/n:.2f} < {a.min_parse} after {n} queries - aborting this model")
                summ["aborted_low_parse"] = True; break
            if n % 50 < len(chunk):
                log(f"{n}/{len(todo)} | {dt:.2f}s/q | parse {n_ok/n:.2f} | acc {n_corr/max(1,n_ok):.2f}")
        f.close()
        summ.update({"n": n, "sec_per_query": round(float(sum(times) / max(1, len(times))), 3), "parse_rate": round(n_ok / max(1, n), 3),
                     "acc_on_parsed": round(n_corr / max(1, n_ok), 3), "input_tokens": n_in if todo else None})

        # optional batched-throughput test on the first items (not written to results)
        if a.batch_test and a.batch_test > 1 and len(P) >= a.batch_test * 3:
            rows = P.to_dict("records")[: a.batch_test * 3]
            try:
                t0 = time.time()
                for i in range(0, len(rows), a.batch_test):
                    texts, _ = R.generate([(load(r["img1"]), load(r["img2"])) for r in rows[i:i + a.batch_test]], a.prompt, a.max_new)
                summ["batch_test"] = {"batch": a.batch_test, "sec_per_query": round((time.time() - t0) / len(rows), 3),
                                      "parse_rate": round(sum(parse_answer(t)["parse_ok"] for t in texts) / len(texts), 3)}
            except Exception as e:
                summ["batch_test"] = {"batch": a.batch_test, "error": f"{type(e).__name__}: {str(e)[:200]}"}
            log(f"batch test: {summ['batch_test']}")
        summ["vram_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        summ["ok"] = True
    except Exception as e:
        summ["error"] = f"{type(e).__name__}: {str(e)[:1500]}"
        log("FAILED: " + summ["error"]); traceback.print_exc()
    summ["wall_s"] = round(time.time() - t_start, 1)
    json.dump(summ, open(a.summary, "w"), indent=2)
    log(f"done: {json.dumps({k: summ.get(k) for k in ['ok', 'n', 'sec_per_query', 'parse_rate', 'acc_on_parsed', 'vram_peak_gb']})}")


if __name__ == "__main__":
    main()


Writing /kaggle/working/n13_output/pdcp_infer.py


## Part A — larger open-weight models (one per GPU)

In [4]:
HF_CACHE = "/tmp/hf_cache"
sys.path.insert(0, OUTPUT_DIR); import pdcp_infer as I; importlib.reload(I)
DEADLINE = T0 + SESSION_HOURS * 3600
def ndone(m):
    f = f"{OUTPUT_DIR}/results/e5/{m}.jsonl"; return sum(1 for _ in open(f)) if os.path.exists(f) else 0
if DRY_RUN:
    rng = np.random.default_rng(0)
    for m in GPU_MODELS + ["Gemini-Flash"]:
        with open(f"{OUTPUT_DIR}/results/e5/{m}.jsonl", "w") as f:
            for r in Q.to_dict("records"):
                a = rng.choice(["1", "2", "same"], p=[0.45, 0.5, 0.05]); p = {"answer": a, "confidence": 4, "reason": "x", "parse_ok": True}
                f.write(json.dumps({"pair_id": r["pair_id"], "order": r["order"], "model": m, "prompt_id": "P-Def", "raw_text": "", "parsed": p, "parse_ok": True, "correct_img": r["correct_img"]}) + "\n")
queue = [m for m in GPU_MODELS if ndone(m) < len(Q) and not DRY_RUN]; running = {}
def launch(model, gpu):
    multi = I.MODELS[model].get("multi", False)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=",".join(str(g) for g in range(N_GPU)) if multi else str(gpu), HF_HUB_CACHE=HF_CACHE, TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    cmd = [sys.executable, f"{OUTPUT_DIR}/pdcp_infer.py", "--model", model, "--pairs", f"{OUTPUT_DIR}/e5_queries.csv",
           "--out", f"{OUTPUT_DIR}/results/e5/{model}.jsonl", "--summary", f"{OUTPUT_DIR}/results/e5/{model}.summary.json",
           "--max_new", "96", "--deadline", str(DEADLINE), "--min_parse", "0.3"]
    logf = open(f"{OUTPUT_DIR}/logs/{model}.log", "a")
    running[gpu] = (model, subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env), logf)
    if multi:
        for g in range(N_GPU):
            if g != gpu: running.setdefault(g, ("(shared)", running[gpu][1], None))
    print(f"[{(time.time()-T0)/60:6.1f} min] start {model} on GPU {'all' if multi else gpu}", flush=True)
def poll():
    for gpu, (model, p, logf) in list(running.items()):
        if p.poll() is not None:
            del running[gpu]
            if logf is None: continue
            logf.close()
            print(f"[{(time.time()-T0)/60:6.1f} min] end {model}: {open(f'{OUTPUT_DIR}/logs/{model}.log').read().strip().splitlines()[-1][-250:]}", flush=True)
            for mid in I.MODELS[model]["ids"]: shutil.rmtree(f"{HF_CACHE}/models--{mid.replace('/', '--')}", ignore_errors=True)
# single-GPU models first, the multi-GPU model when both GPUs are free
queue.sort(key=lambda m: I.MODELS[m].get("multi", False))
for gpu in range(N_GPU):
    if queue and gpu not in running and (not I.MODELS[queue[0]].get("multi") or not running): launch(queue.pop(0), gpu)
print("GPU models running in the background; Part B starts now.")

[   0.6 min] start Pixtral-12B on GPU all
GPU models running in the background; Part B starts now.


## Part B — Gemini Flash through the API (optional; runs while the GPU models work)

In [5]:
def gemini_key():
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        return os.environ.get("GEMINI_API_KEY")
KEY = gemini_key() if (RUN_API and not DRY_RUN) else None
api_note = "skipped (no GEMINI_API_KEY secret)" if not KEY else ""
if KEY:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=KEY)
    names = [m.name.split("/")[-1] for m in client.models.list()]
    if GEMINI_MODEL == "auto":
        cand = [n for n in names if "flash" in n and not any(x in n for x in ["lite", "image", "tts", "audio", "live", "embed", "exp", "thinking"])]
        ver = lambda n: tuple(int(x) for x in re.findall(r"\d+", n)[:2]) + ((0,) if "preview" in n else (1,))
        cand = sorted(cand, key=ver, reverse=True); print("Flash models available:", cand)
        GEMINI_MODEL = cand[0] if cand else None
    print("Gemini model:", GEMINI_MODEL)
    out = f"{OUTPUT_DIR}/results/e5/Gemini-Flash.jsonl"
    done = set()
    if os.path.exists(out):
        for l in open(out):
            try: r = json.loads(l); done.add((r["pair_id"], r["order"]))
            except Exception: pass
    todo = [r for r in Q.to_dict("records") if (r["pair_id"], r["order"]) not in done]
    print(len(done), "done earlier |", len(todo), "to do | this run: up to", API_MAX_PER_RUN)
    n_run, last, stop = 0, 0.0, False
    with open(out, "a") as f:
        for r in todo:
            if n_run >= API_MAX_PER_RUN or time.time() > DEADLINE - 900 or stop: break
            wait = 60.0 / API_RPM - (time.time() - last)
            if wait > 0: time.sleep(wait)
            contents = [I.SYSTEM, "Image 1:", Image.open(r["img1"]).convert("RGB"), "Image 2:", Image.open(r["img2"]).convert("RGB"), I.PROMPTS["P-Def"]]
            text, err = None, None
            for attempt in range(4):
                last = time.time()
                try:
                    resp = client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=types.GenerateContentConfig(temperature=0, max_output_tokens=1024))
                    text = resp.text or ""; break
                except Exception as e:
                    err = f"{type(e).__name__}: {str(e)[:300]}"
                    if "per day" in err.lower() or "PerDay" in err: stop = True; break
                    if "429" in err or "RESOURCE_EXHAUSTED" in err or "503" in err: time.sleep(30 * (attempt + 1)); continue
                    break
            if text is None:
                print("request failed:", err);
                if stop: print("daily limit reached - rerun later with RESUME_ROOT");
                continue
            p = I.parse_answer(text)
            f.write(json.dumps({"pair_id": r["pair_id"], "order": r["order"], "model": "Gemini-Flash", "model_id": GEMINI_MODEL, "prompt_id": "P-Def",
                                "raw_text": text, "parsed": p, "parse_ok": p["parse_ok"], "correct_img": r["correct_img"],
                                "correct": (p["answer"] == r["correct_img"]) if p["parse_ok"] else None, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}) + "\n"); f.flush()
            n_run += 1
            if n_run % 50 == 0: print(f"  Gemini {n_run} this run | {(time.time()-T0)/60:.0f} min", flush=True)
    api_note = f"{GEMINI_MODEL}: {n_run} requests this run, {len(done) + n_run}/{len(Q)} total"
print("Part B:", api_note)

Part B: skipped (no GEMINI_API_KEY secret)


In [6]:
# wait for the GPU models
last = 0
while queue or running:
    for gpu in range(N_GPU):
        if gpu not in running and queue and time.time() < DEADLINE - 1800 and (not I.MODELS[queue[0]].get("multi") or not running): launch(queue.pop(0), gpu)
    if not running: break
    time.sleep(30); poll()
    if time.time() - last > 1800: last = time.time(); print("   " + " | ".join(f"{m}: {ndone(m)}/{len(Q)}" for m in GPU_MODELS), flush=True)
status = {m: {"done": ndone(m), "total": len(Q)} for m in GPU_MODELS + ["Gemini-Flash"]}; status["api"] = api_note
for m in GPU_MODELS:
    f = f"{OUTPUT_DIR}/results/e5/{m}.jsonl"
    if os.path.exists(f): status[m]["downscaled_512px"] = sum(json.loads(l).get("downscaled", False) for l in open(f))
status["all_complete"] = all(status[m]["done"] >= len(Q) for m in GPU_MODELS)
json.dump(status, open(f"{OUTPUT_DIR}/e5_status.json", "w"), indent=2); print(json.dumps(status, indent=1))

   InternVL-14B: 1168/1168 | Pixtral-12B: 0/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 58/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 124/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 189/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 255/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 321/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 387/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 453/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 518/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 584/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 650/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 716/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 782/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 848/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 913/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 979/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 1045/1168
   InternVL-14B: 1168/1168 | Pixtral-12B: 1111/1168
[ 537.1 min] end Pixtral-12B: [Pixtral-12B 19:27:40] done: {"ok": true, "n": 1168, 

## Analysis (same measures as E4b, plus the E1/E2 models on the same 200 pairs)

In [7]:
def load(root, sub):
    R = []
    for f in glob.glob(f"{root}/**/results/{sub}/*.jsonl", recursive=True): R += [json.loads(l) for l in open(f)]
    R = pd.DataFrame(R)
    if not len(R): return R
    a = R.parsed.map(lambda p: (p or {}).get("answer")); R["ans"] = a
    R["choice"] = np.where(a == "same", "same", np.where(a.isin(["1", "2"]), np.where((a == "1") == (R.order == "AB"), "A", "B"), None))
    R["pa"] = R.choice.map({"A": 1.0, "B": 0.0, "same": 0.5}); return R
E5 = load(OUTPUT_DIR, "e5"); E1 = load(N05_ROOT, "e1"); E2 = load(N06_ROOT, "e2")
Bi = BASE.set_index("pair_id"); Mi = MEL.set_index("pair_id"); key = dict(zip(PR.pair_id, PR.correct))
Di = DR.set_index("drawing_id")
def mel_of(did):
    r = Di.loc[did]; gc = U.GraphCache(U.make_graph(r.family, int(r.n), int(r.graph_seed))); return U.mean_edge_length(np.array(r.coords).reshape(-1, 2), gc.E)
MELV = {d: mel_of(d) for d in set(BASE.drawing_A) | set(BASE.drawing_B)}
dmel = pd.Series({p: MELV[Bi.loc[p, "drawing_B"]] - MELV[Bi.loc[p, "drawing_A"]] for p in Bi.index})
def summarize(m, R):
    b = R[R.pair_id.isin(Bi.index)]; ok = b[b.choice.notna()]; nz = ok[ok.pair_id.map(Bi.dstress_design) > 0]
    pa = ok.groupby("pair_id").pa.mean(); z = pa[pa.index.map(Bi.dstress_design) == 0]; zc = z.index.map(Bi.cond)
    out = {"model": m, "family": FAMILY.get(m, "?"), "params_B": PARAMS_B.get(m, np.nan), "quant": "api" if m == "Gemini-Flash" else ("nf4" if m in ("Q25VL-7B", "InternVL-14B", "Pixtral-12B") else "fp16"),
           "n": len(b), "parse": b.choice.notna().mean() if len(b) else np.nan, "acc_dstress>0": (nz.choice == nz.pair_id.map(key)).mean() if len(nz) else np.nan,
           "p_image1": (ok.ans == "1").mean() if len(ok) else np.nan, "p_same": (ok.ans == "same").mean() if len(ok) else np.nan,
           "lure_gap": z[zc == "congruent"].mean() - z[zc == "conflict_all"].mean() if len(z) else np.nan}
    mm = R[R.pair_id.isin(Mi.index) & R.choice.notna()]
    if len(mm) and len(pa):
        pm = mm.groupby("pair_id").pa.mean(); sh = (pm - pm.index.map(Mi.base_pair).map(pa)).dropna(); dl = sh.index.map(Mi.cf_delta)
        for d in MEL_DELTAS: out[f"mel_shift_{d:+.2f}"] = sh[dl == d].mean()
        out["compactness_bias"] = out[f"mel_shift_{MEL_DELTAS[0]:+.2f}"] - out[f"mel_shift_{MEL_DELTAS[1]:+.2f}"]
    X = pd.DataFrame({"ksm": Bi.d_ksm, "nu": Bi.d_nu, "eld": Bi.d_eld, "xr": Bi.d_xr, "mel": dmel}).loc[pa.index.intersection(Bi.index)]
    if len(X) > 30:
        Z = (X - X.mean()) / X.std(); y = pa[X.index]; y = (y - y.mean()) / (y.std() + 1e-12)
        bt = np.linalg.lstsq(np.c_[np.ones(len(Z)), Z.values], y.values, rcond=None)[0][1:]; out.update({f"beta_{k}": v for k, v in zip(X.columns, bt)})
        out["cue_share"] = sum(abs(out[f"beta_{k}"]) for k in ["nu", "eld", "xr", "mel"]) / (sum(abs(out[f"beta_{k}"]) for k in ["nu", "eld", "xr", "mel"]) + abs(out["beta_ksm"]))
    return out
rows = [{**summarize(m, g), "source": "N13"} for m, g in E5.groupby("model")] if len(E5) else []
for m in sorted(set(E1.model) if len(E1) else []):
    if m in PARAMS_B: rows.append({**summarize(m, pd.concat([E1[E1.model == m], E2[E2.model == m] if len(E2) else E1.iloc[:0]])), "source": "E1/E2"})
SS = pd.DataFrame(rows)
p2 = 1 - SS.p_image1 - SS.p_same
SS["response_policy"] = np.select([SS.parse < 0.05, SS.p_same >= 0.5, SS.p_image1 >= 0.9, p2 >= 0.9], ["no answer", "mostly 'same'", "always 'Image 1'", "always 'Image 2'"], "discriminating")
SS.round(4).to_csv(f"{OUTPUT_DIR}/e5_summary.csv", index=False); pd.set_option("display.width", 220); print(SS.round(3).to_string(index=False))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, col, t in zip(ax, ["acc_dstress>0", "cue_share", "compactness_bias"], ["Accuracy (Δstress > 0)", "Cue share", "Compactness bias"]):
    if col not in SS: continue
    for _, r in SS.iterrows():
        x = r.params_B if np.isfinite(r.params_B) else 18
        a.scatter(x, r[col], s=50, marker="*" if r.model == "Gemini-Flash" else "o", facecolor="white" if r.response_policy != "discriminating" else None)
        a.annotate(r.model, (x, r[col]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    a.set_xlabel("parameters (B)  (star at right: proprietary)"); a.set_title(t)
ax[0].axhline(0.5, ls=":", c="grey"); ax[2].axhline(0, c="k", lw=.8)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/fig_e5_scale.png", dpi=130); plt.show()
shutil.rmtree(f"{OUTPUT_DIR}/__pycache__", ignore_errors=True)
lines = [f"N13 larger models · {(time.time()-T0)/3600:.2f} h · " + " | ".join(f"{m}: {status[m]['done']}/{status[m]['total']}" for m in GPU_MODELS + ['Gemini-Flash']) + f" · API: {api_note}", ""]
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f); lines.append(f"{os.path.relpath(p, OUTPUT_DIR):40s} {os.path.getsize(p)/1024:9.1f} KB")
open(f"{OUTPUT_DIR}/MANIFEST.txt", "w").write("\n".join(lines)); print("\n".join(lines))

       model      family  params_B quant   n  parse  acc_dstress>0  p_image1  p_same  lure_gap  mel_shift_-0.05  mel_shift_+0.10  compactness_bias  beta_ksm  beta_nu  beta_eld  beta_xr  beta_mel  cue_share source response_policy
InternVL-14B InternVL3.5        14   nf4 400    1.0          0.522     0.120   0.000     0.062            0.087           -0.072             0.159    -0.055   -0.172     0.251   -0.114     0.195      0.931    N13  discriminating
 Pixtral-12B     Pixtral        12   nf4 400    1.0          0.562     0.622   0.002     0.250            0.104           -0.073             0.177     0.127   -0.285     0.288    0.009    -0.079      0.839    N13  discriminating
 InternVL-4B InternVL3.5         4  fp16 400    1.0          0.565     0.560   0.000     0.542            0.129           -0.137             0.266     0.063    0.133     0.214    0.030     0.118      0.888  E1/E2  discriminating
    Q25VL-7B  Qwen2.5-VL         7   nf4 400    1.0          0.450     0.280   0.000